# CS 435 — Project 3: Medical X-Ray Anomaly Detection
## Phase 4: CNN Retrained on GAN-Augmented Data

**Goal of this notebook:**  
Retrain the **identical CNN architecture** from Phase 2 on the augmented training set produced in Phase 3 (`train_augmented.csv` = real NIH images + GAN-generated synthetic minority-class X-rays). Then evaluate on the **same held-out test set** used in Phase 2 and save metrics to a CSV that Phase 5 will use for side-by-side comparison.

### Why the same architecture?
For the comparison to be scientifically valid, **only one variable should change** between Phase 2 and Phase 4: the training data. The network architecture, optimizer, loss function, callbacks, and test set must all be identical. If we changed the architecture too, we would not know whether any improvement came from the GAN augmentation or from the architecture change.

This is the standard experimental design principle that Chollet refers to throughout *Deep Learning with Python* — change one thing at a time and measure the effect.

**Prerequisites:** Phases 1, 2, and 3 notebooks must have been run.

---
## Step 1: Mount Drive and import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, classification_report, roc_curve
)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## Step 2: Configuration

All settings **must be identical to Phase 2** to ensure a fair comparison.
The only things that differ from Phase 2:
- We load `train_augmented.csv` instead of `train.csv`
- The model checkpoint saves to a different filename
- Metrics are tagged `augmented_cnn` instead of `baseline_cnn`

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath("")
SPLITS_DIR  = os.path.join(BASE_DIR, 'splits')
MODEL_DIR   = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

IMG_SIZE    = 224
CHANNELS    = 1
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)
TARGET_CLASSES = ['No Finding', 'Atelectasis', 'Effusion', 'Cardiomegaly', 'Pneumonia']
NUM_CLASSES    = len(TARGET_CLASSES)

BATCH_SIZE    = 32
EPOCHS        = 30
LEARNING_RATE = 1e-3
PATIENCE      = 5
THRESHOLD     = 0.5

---
## Step 3: Load the augmented train split + original val/test splits

- **Train:** `train_augmented.csv` — real NIH images + GAN synthetic images from Phase 3
- **Validation:** `val.csv` — original Phase 1 split, real images only (never augmented)
- **Test:** `test.csv` — original Phase 1 split, real images only (never augmented)

Keeping validation and test sets real-only is critical: we want to measure how well the model performs on real, unseen X-rays — not on synthetic ones.

In [ ]:
# Augmented training data (real + GAN synthetic)
aug_train_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train_augmented.csv'))

# Unchanged validation and test sets (real only)
val_df  = pd.read_csv(os.path.join(SPLITS_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(SPLITS_DIR, 'test.csv'))

print(f"Augmented train : {len(aug_train_df):,} images")
print(f"  - Real        : {(~aug_train_df['is_synthetic']).sum():,}")
print(f"  - Synthetic   : {aug_train_df['is_synthetic'].sum():,}")
print(f"Validation      : {len(val_df):,} images (real only)")
print(f"Test            : {len(test_df):,} images (real only)")

print("\nAugmented train class distribution:")
print(aug_train_df[TARGET_CLASSES].sum().sort_values(ascending=False))

---
## Step 4: Rebuild the preprocessing pipeline

The preprocessing function is identical to Phase 1 and Phase 2. Both real and synthetic images go through the same pipeline:
- Load PNG from disk
- Decode to grayscale
- Resize to 224×224 (upsample from the GAN's 64×64 output)
- Normalize to [0, 1]

The resize step handles the GAN's lower resolution output transparently — the CNN never knows whether it's looking at a real 1024×1024 downsampled image or a GAN-generated 64×64 upsampled image.

In [ ]:
def preprocess_image(filepath, label):
    """Identical to Phase 1 and Phase 2 preprocessing."""
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_png(raw, channels=CHANNELS)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def build_dataset(dataframe, shuffle=False):
    filepaths = dataframe['filepath'].values
    labels    = dataframe[TARGET_CLASSES].values.astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=42)
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# Training uses augmented data; val and test use original real data
train_dataset = build_dataset(aug_train_df, shuffle=True)
val_dataset   = build_dataset(val_df,       shuffle=False)
test_dataset  = build_dataset(test_df,      shuffle=False)

print(f"Train batches : {len(train_dataset)}  (augmented)")
print(f"Val batches   : {len(val_dataset)}   (real only)")
print(f"Test batches  : {len(test_dataset)}   (real only)")

---
## Step 5: Build the CNN — same architecture as Phase 2

This is a verbatim copy of the `build_cnn()` function from Phase 2. **Do not change anything here.** The architecture must be identical for the comparison to be valid.

Quick recap of the architecture:
- 3 Conv blocks: Conv2D → BatchNorm → ReLU → MaxPool → Dropout(0.25)
- Filter depth doubles each block: 32 → 64 → 128
- GlobalAveragePooling2D → Dense(256) → Dropout(0.5)
- Dense(5, sigmoid) — multi-label output
- Loss: binary_crossentropy (correct for multi-label as per Chollet Ch. 14)

In [ ]:
def build_cnn(input_shape, num_classes):
    """
    IDENTICAL architecture to Phase 2. Do not modify.
    Any change here would invalidate the baseline comparison.
    """
    inputs = keras.Input(shape=input_shape, name='xray_input')

    # Conv Block 1: 224×224 → 112×112
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False, name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu', name='relu1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = layers.Dropout(0.25, name='drop1')(x)

    # Conv Block 2: 112×112 → 56×56
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False, name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu', name='relu2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = layers.Dropout(0.25, name='drop2')(x)

    # Conv Block 3: 56×56 → 28×28
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False, name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu', name='relu3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    x = layers.Dropout(0.25, name='drop3')(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D(name='gap')(x)

    # Dense Head
    x = layers.Dense(256, use_bias=False, name='fc1')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.Activation('relu', name='relu4')(x)
    x = layers.Dropout(0.5, name='drop4')(x)

    # Multi-label output: sigmoid, one neuron per class
    outputs = layers.Dense(num_classes, activation='sigmoid',
                           name='predictions')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='xray_cnn_augmented')


model = build_cnn(INPUT_SHAPE, NUM_CLASSES)
model.summary()

---
## Step 6: Compile — identical to Phase 2

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy', threshold=THRESHOLD),
        keras.metrics.AUC(name='auc', multi_label=True),
        keras.metrics.Precision(name='precision', thresholds=THRESHOLD),
        keras.metrics.Recall(name='recall',    thresholds=THRESHOLD),
    ]
)
print("Model compiled — identical settings to Phase 2.")

---
## Step 7: Callbacks — identical to Phase 2 (different save path)

In [ ]:
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'augmented_cnn_best.keras')

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.CSVLogger(
        os.path.join(RESULTS_DIR, 'augmented_training_log.csv')
    )
]

print(f"Callbacks ready. Best model will save to:\n  {CHECKPOINT_PATH}")

---
## Step 8: Train the augmented CNN

Training on the augmented dataset takes longer per epoch than Phase 2 because there are more images (real + synthetic). Expect roughly 1.3–1.5× the per-epoch time of Phase 2 on a free Colab T4 GPU.

The hypothesis is: with more balanced class representation, the CNN will achieve higher AUC and F1 on the minority classes (`Pneumonia`, `Cardiomegaly`) compared to the Phase 2 baseline.

In [ ]:
print("Starting augmented CNN training...")
print(f"Training on {len(aug_train_df):,} images ({aug_train_df['is_synthetic'].sum():,} synthetic)")
print("-" * 60)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining complete — ran for {len(history.history['loss'])} epochs.")

---
## Step 9: Plot training curves

We overlay the Phase 2 baseline training log (loaded from Drive) on top of the Phase 4 curves so we can see directly whether augmentation helped the model converge to a better validation AUC and lower loss.

In [ ]:
# Load Phase 2 baseline training log for comparison
baseline_log_path = os.path.join(RESULTS_DIR, 'baseline_training_log.csv')
augmented_log = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss ──────────────────────────────────────────────────────────────────────
axes[0].plot(augmented_log['val_loss'], label='Augmented val loss',
             color='#1565C0', linewidth=2)
axes[0].plot(augmented_log['loss'],     label='Augmented train loss',
             color='#1565C0', linewidth=1, linestyle='--', alpha=0.6)

if os.path.exists(baseline_log_path):
    baseline_log = pd.read_csv(baseline_log_path)
    axes[0].plot(baseline_log['val_loss'], label='Baseline val loss',
                 color='#F44336', linewidth=2, alpha=0.7)

axes[0].set_title('Validation Loss — Baseline vs Augmented', fontsize=11)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# ── AUC ───────────────────────────────────────────────────────────────────────
axes[1].plot(augmented_log['val_auc'], label='Augmented val AUC',
             color='#1565C0', linewidth=2)
axes[1].plot(augmented_log['auc'],    label='Augmented train AUC',
             color='#1565C0', linewidth=1, linestyle='--', alpha=0.6)

if os.path.exists(baseline_log_path):
    axes[1].plot(baseline_log['val_auc'], label='Baseline val AUC',
                 color='#F44336', linewidth=2, alpha=0.7)

axes[1].set_title('Validation AUC-ROC — Baseline vs Augmented', fontsize=11)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC-ROC')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('Phase 2 Baseline vs Phase 4 GAN-Augmented CNN — Training Curves', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'phase2_vs_phase4_training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Evaluate on the test set

In [ ]:
best_model = keras.models.load_model(CHECKPOINT_PATH)
print(f"Loaded best augmented model from: {CHECKPOINT_PATH}")

print("\nEvaluating on test set (real images only)...")
test_results = best_model.evaluate(test_dataset, verbose=1)

print("\n=== Augmented CNN — Test Set Results ===")
for name, value in zip(best_model.metrics_names, test_results):
    print(f"  {name:15s}: {value:.4f}")

---
## Step 11: Collect raw predictions and compute per-class metrics

In [ ]:
# Collect all predictions over the test set
y_true_list, y_prob_list = [], []
for images, labels in test_dataset:
    probs = best_model(images, training=False)
    y_true_list.append(labels.numpy())
    y_prob_list.append(probs.numpy())

y_true = np.vstack(y_true_list)
y_prob = np.vstack(y_prob_list)
y_pred = (y_prob >= THRESHOLD).astype(int)

# Per-class AUC-ROC
print("=== Per-Class AUC-ROC (Augmented CNN — With GAN) ===")
auc_scores = {}
for i, cls in enumerate(TARGET_CLASSES):
    auc = roc_auc_score(y_true[:, i], y_prob[:, i])
    auc_scores[cls] = auc
    print(f"  {cls:20s}: AUC = {auc:.4f}")

macro_auc = np.mean(list(auc_scores.values()))
print(f"\n  {'Macro Average':20s}: AUC = {macro_auc:.4f}")

print("\n=== Per-Class Classification Report ===")
print(classification_report(y_true, y_pred,
                             target_names=TARGET_CLASSES, zero_division=0))

---
## Step 12: Save augmented CNN metrics to CSV

We save in the same format as Phase 2's `baseline_metrics.csv`. Phase 5 will load both files and merge them for the final comparison table and plots.

In [ ]:
metrics_rows = []
for i, cls in enumerate(TARGET_CLASSES):
    metrics_rows.append({
        'class':     cls,
        'auc_roc':   roc_auc_score(y_true[:, i], y_prob[:, i]),
        'f1':        f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
        'precision': precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
        'recall':    recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
        'model':     'augmented_cnn'
    })

metrics_rows.append({
    'class':     'MACRO_AVG',
    'auc_roc':   macro_auc,
    'f1':        f1_score(y_true, y_pred, average='macro', zero_division=0),
    'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
    'recall':    recall_score(y_true, y_pred, average='macro', zero_division=0),
    'model':     'augmented_cnn'
})

aug_metrics_df = pd.DataFrame(metrics_rows)
aug_metrics_path = os.path.join(RESULTS_DIR, 'augmented_metrics.csv')
aug_metrics_df.to_csv(aug_metrics_path, index=False)

print(f"Augmented metrics saved to: {aug_metrics_path}")
print("\n=== Augmented CNN Metrics ===")
print(aug_metrics_df.to_string(index=False))

---
## Step 13: Side-by-side quick comparison — Phase 2 vs Phase 4

We load both metric CSVs here for a quick in-notebook look. The full comparison plots and statistical summary are in Phase 5.

In [ ]:
baseline_metrics_path = os.path.join(RESULTS_DIR, 'baseline_metrics.csv')

if os.path.exists(baseline_metrics_path):
    base_df = pd.read_csv(baseline_metrics_path)
    aug_df  = aug_metrics_df.copy()

    # Merge on class for comparison
    comparison = base_df.merge(aug_df, on='class', suffixes=('_baseline', '_augmented'))

    # Compute delta (improvement)
    comparison['auc_delta'] = comparison['auc_roc_augmented'] - comparison['auc_roc_baseline']
    comparison['f1_delta']  = comparison['f1_augmented']      - comparison['f1_baseline']

    print("=== AUC-ROC Comparison: Baseline vs GAN-Augmented ===")
    display_cols = ['class', 'auc_roc_baseline', 'auc_roc_augmented', 'auc_delta']
    print(comparison[display_cols].to_string(index=False))

    print("\n=== F1-Score Comparison ===")
    display_cols_f1 = ['class', 'f1_baseline', 'f1_augmented', 'f1_delta']
    print(comparison[display_cols_f1].to_string(index=False))

    print("\n🔑 Key: positive delta = improvement from GAN augmentation")
    print("Focus on Pneumonia and Cardiomegaly — these should show the largest gains.")
else:
    print("Baseline metrics CSV not found — run Phase 2 first.")
    print("Augmented metrics saved. Run Phase 2 and then Phase 5 for the full comparison.")

---
## Step 14: Quick ROC curve overlay — all classes, both models

In [ ]:
# Load Phase 2 baseline model to regenerate its ROC curves for overlay
baseline_model_path = os.path.join(MODEL_DIR, 'baseline_cnn_best.keras')

if os.path.exists(baseline_model_path):
    baseline_model = keras.models.load_model(baseline_model_path)

    # Collect baseline predictions on the same test set
    y_prob_base_list = []
    for images, _ in test_dataset:
        y_prob_base_list.append(baseline_model(images, training=False).numpy())
    y_prob_base = np.vstack(y_prob_base_list)

    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 4))
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

    for i, (cls, ax, color) in enumerate(zip(TARGET_CLASSES, axes, colors)):
        # Baseline ROC
        fpr_b, tpr_b, _ = roc_curve(y_true[:, i], y_prob_base[:, i])
        auc_b = roc_auc_score(y_true[:, i], y_prob_base[:, i])

        # Augmented ROC
        fpr_a, tpr_a, _ = roc_curve(y_true[:, i], y_prob[:, i])
        auc_a = auc_scores[cls]

        ax.plot(fpr_b, tpr_b, color=color, lw=1.5, linestyle='--',
                alpha=0.7, label=f'Baseline AUC={auc_b:.3f}')
        ax.plot(fpr_a, tpr_a, color=color, lw=2.5,
                label=f'Augmented AUC={auc_a:.3f}')
        ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.4)
        ax.set_title(cls, fontsize=10)
        ax.set_xlabel('FPR', fontsize=8)
        ax.set_ylabel('TPR', fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.suptitle('ROC Curves: Baseline CNN vs GAN-Augmented CNN (solid = augmented)',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'phase2_vs_phase4_roc_overlay.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Phase 2 baseline model not found. Skipping ROC overlay.")
    print("Run Phase 2 first, then re-run this cell.")

---
## Phase 4 Summary

| Step | What we did | Output |
|------|-------------|--------|
| 1–2  | Setup + config (identical to Phase 2) | Environment ready |
| 3    | Loaded `train_augmented.csv` (real + GAN synthetic) | Augmented training data |
| 4    | Rebuilt preprocessing pipeline (identical to Phase 2) | `train/val/test_dataset` |
| 5    | Built CNN (identical architecture to Phase 2) | `xray_cnn_augmented` model |
| 6–7  | Compiled + callbacks (identical to Phase 2) | Training ready |
| 8    | Trained on augmented data | `augmented_cnn_best.keras` |
| 9    | Overlaid training curves vs Phase 2 baseline | `phase2_vs_phase4_training_curves.png` |
| 10–11| Evaluated on held-out test set, per-class metrics | AUC, F1, precision, recall |
| 12   | Saved metrics CSV for Phase 5 | `augmented_metrics.csv` |
| 13   | Quick delta table (baseline vs augmented) | Inline comparison |
| 14   | ROC curve overlay — both models on same axes | `phase2_vs_phase4_roc_overlay.png` |

### What to look for in your results:

The key question is: **did GAN augmentation improve the AUC and F1 for the minority classes?**

You should see:
- **Pneumonia and Cardiomegaly AUC/F1 increase** — these are the classes the GAN synthesized extra data for
- **Effusion and No Finding roughly unchanged** — the GAN didn't augment these, so no change is expected
- Possibly a slight **overall macro AUC increase** if minority class improvement outweighs any majority class degradation

If the minority class metrics did NOT improve, possible reasons to discuss in your report:
- The GAN needed more training epochs (try 500+)
- The generated images suffered from mode collapse (too repetitive, not diverse enough)
- `N_SYNTHETIC_PER_CLASS` was insufficient — try doubling it

**Next step → Phase 5:** Load both metric CSVs and produce the final comparison figures and tables for your project report.